In [ ]:
# ==================================================
# process_mmgbsa_data.py
#
# Purpose:
#   Process MM/GBSA energy data from multiple systems
#   and multiple replicas.
#
# Input:
#   final_results_rep1.dat
#
# Output:
#   mmgbsa_analysis_data.npz
#
# Workflow:
#   MM/GBSA .dat files
#   ↓
#   Read energy values
#   ↓
#   Align replica length
#   ↓
#   Calculate mean and standard deviation
#   ↓
#   Save as NPZ file
# ==================================================

import os
import numpy as np


# ==================================================
# Part 1. User Settings
# ==================================================
#
# Most users only need to modify this section.
#
# ==================================================

# Main working directory

BASE_PATH = (
    "/ceph/sharedfs/work/MYTLab/asher/"
    "center_ion_project/na/mmgbsa_control_12_15/"
)


# System definitions
#
# folder:
#   Common folder prefix.
#
# Example:
#   folder = "mmgbsa606"
#
# The script will search:
#   mmgbsa606_rep1
#   mmgbsa606_rep2
#   mmgbsa606_rep3

SYSTEMS = [

    {
        "folder": "mmgbsa606",
        "label": "Docking score=6.06",
        "color": "royalblue",
    },

    {
        "folder": "mmgbsa687",
        "label": "Docking score=6.87",
        "color": "seagreen",
    },

    {
        "folder": "mmgbsa688",
        "label": "Docking score=6.88",
        "color": "darkorange",
    },

    {
        "folder": "mmgbsa688_2",
        "label": "Docking score=6.88_2",
        "color": "crimson",
    },

    {
        "folder": "mmgbsa721",
        "label": "Docking score=7.21",
        "color": "purple",
    },

]


# Replica folder suffixes

REP_SUFFIXES = [
    "_rep1",
    "_rep2",
    "_rep3",
]


# Energy result filename inside each replica folder
#
# Note:
#   This filename must exist inside every replica folder.
#
# Example:
#   BASE_PATH/mmgbsa606_rep1/final_results_rep1.dat

DATA_FILENAME = "final_results_rep1.dat"


# Time conversion
#
# If 500 frames = 1 ns:
#   time = frame_index / 500

FRAMES_PER_NS = 500.0


# Output file

OUTPUT_FILE = "mmgbsa_analysis_data.npz"


# ==================================================
# Part 2. Function: Read Energy File
# ==================================================

def parse_energy_file(file_path):
    """
    Read MM/GBSA energy file and extract the second column.

    Expected file format:
        column 1 = frame index
        column 2 = energy value

    Lines starting with # or @ will be skipped.
    """

    values = []

    if not os.path.exists(file_path):

        print(
            f"  [Warning] File not found: {file_path}"
        )

        return None

    try:

        with open(file_path, "r") as file:

            for line in file:

                line = line.strip()

                # Skip empty lines and comment lines

                if not line:
                    continue

                if line.startswith(("#", "@")):
                    continue

                parts = line.split()

                if len(parts) < 2:
                    continue

                try:

                    energy_value = float(parts[1])

                    values.append(energy_value)

                except ValueError:

                    continue

    except Exception as error:

        print(
            f"  [Error] Failed to read file: {file_path}"
        )

        print(
            f"  Reason: {error}"
        )

        return None

    return np.array(values)


# ==================================================
# Part 3. Main Processing
# ==================================================

def main():

    save_dict = {}

    print("====================================")
    print("MM/GBSA data processing started")
    print(f"Base path: {BASE_PATH}")
    print("====================================\n")


    # ----------------------------------------------
    # Process each system
    # ----------------------------------------------

    for system_id, system_info in enumerate(SYSTEMS):

        folder_prefix = system_info["folder"]
        label = system_info["label"]
        color = system_info["color"]

        print(
            f"Processing system: {label}"
        )

        print(
            f"Folder prefix: {folder_prefix}"
        )

        rep_data_list = []


        # ------------------------------------------
        # Process each replica
        # ------------------------------------------

        for suffix in REP_SUFFIXES:

            current_folder = (
                f"{folder_prefix}{suffix}"
            )

            file_path = os.path.join(
                BASE_PATH,
                current_folder,
                DATA_FILENAME
            )

            data = parse_energy_file(
                file_path
            )

            if data is not None and len(data) > 0:

                rep_data_list.append(data)

                print(
                    f"  Loaded {suffix}: "
                    f"{len(data)} frames"
                )

            else:

                print(
                    f"  Skipped {suffix}: no valid data"
                )


        # ------------------------------------------
        # Skip system if no valid replica is found
        # ------------------------------------------

        if not rep_data_list:

            print(
                f"  [Error] No valid data for {label}"
            )

            print(
                "  This system will be skipped.\n"
            )

            continue


        # ------------------------------------------
        # Align replica length
        # ------------------------------------------
        #
        # Different replicas may have slightly different
        # frame numbers.
        #
        # To calculate mean and std correctly, all replicas
        # are trimmed to the shortest length.
        #
        # ------------------------------------------

        min_len = min(
            len(data)
            for data in rep_data_list
        )

        trimmed_data = np.array([

            data[:min_len]

            for data in rep_data_list

        ])


        # ------------------------------------------
        # Calculate statistics
        # ------------------------------------------

        mean_values = np.mean(
            trimmed_data,
            axis=0
        )

        std_values = np.std(
            trimmed_data,
            axis=0
        )


        # ------------------------------------------
        # Create time axis
        # ------------------------------------------

        time_axis = (
            np.arange(min_len)
            / FRAMES_PER_NS
        )


        # ------------------------------------------
        # Store processed data
        # ------------------------------------------

        key_prefix = f"sys_{system_id}"

        save_dict[f"{key_prefix}_time"] = time_axis

        save_dict[f"{key_prefix}_mean"] = mean_values

        save_dict[f"{key_prefix}_std"] = std_values

        save_dict[f"{key_prefix}_label"] = label

        save_dict[f"{key_prefix}_color"] = color

        print(
            f"  Statistics calculated. "
            f"Frames used: {min_len}\n"
        )


    # ----------------------------------------------
    # Save processed data
    # ----------------------------------------------

    if save_dict:

        np.savez(
            OUTPUT_FILE,
            **save_dict
        )

        print("====================================")
        print("MM/GBSA data processing completed")
        print(f"Output file: {os.path.abspath(OUTPUT_FILE)}")
        print("Use this file for plotting.")
        print("====================================")

    else:

        print("====================================")
        print("[Error] No data saved.")
        print("Please check input paths and filenames.")
        print("====================================")


# ==================================================
# Part 4. Run Script
# ==================================================

if __name__ == "__main__":

    main()